In [0]:
# ======================================
# Dataset: olist_orders
# Layer: Silver
# Source: Bronze (delta)
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format('delta').load(BRONZE_ORDERS_PATH)

In [0]:
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
fmt = "yyyy-MM-dd HH:mm:ss"
timestamp_cols = {
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
}

df = df.select(
    *[
        F.to_timestamp(c, fmt).alias(c) if c in timestamp_cols else F.col(c)
        for c in df.columns
    ]
)



In [0]:
df = df.filter(
    F.col("order_id").isNotNull() &
    F.col("customer_id").isNotNull()
)

In [0]:
VALID_STATUS = [
    "delivered",
    "shipped",
    "canceled",
    "processing",
    "approved",
    "invoiced"
]

df = df.filter(F.col("order_status").isin(VALID_STATUS))


In [0]:
df = df.filter(
    (F.col("order_approved_at").isNull()) |
    (F.col("order_purchase_timestamp") <= F.col("order_approved_at"))
)


In [0]:
df.printSchema()

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
write_silver(df, SILVER_ORDERS_PATH)